Processar o arquivo de base com as perguntas.

In [138]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-05 18:11:56--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9811 (9.6K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]   9.58K  --.-KB/s    in 0s      

2026-04-05 18:11:56 (80.9 MB/s) - ‘questions.ipynb’ saved [9811/9811]



Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [141]:
# Instalar ou atualizar as bibliotecas necessárias.
!pip install -q -U openai bert-score
# !pip install -q -U google-genai google-auth

# Importar bibliotecas.
import pandas as pd
import os
from openai import OpenAI
from google import genai
from google.colab import userdata
from bert_score import score

Funcões dos clientes das IAs, modelos e realização de consultas.

In [142]:
# Iniciar o cliente da IA
def client_ai_instance(name_api_key):
  # Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
  # O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
  # Tal chave previamente criada no Google AI Studio ou no Console do Groq.
  # Observação: o nome da chave definido precisa ser o mesmo inclusive com diferenciação de letra maiúscula e minúscula.
  api_key = userdata.get(name_api_key)

  # Verificar se foi o groq ou google.
  if name_api_key == 'groq_api_key':
    client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=api_key
  )
  elif name_api_key == 'oprouter_api_key':
    client_ai = OpenAI(
      base_url="https://openrouter.ai/api/v1/",
      api_key=api_key
    )
  elif name_api_key == 'google_api_key':
    client_ai = genai.Client(api_key=api_key)
  else:
    print('chave não encontrada')

  # Retornar instancia do cliente da IA.
  return client_ai

  # Inicializar o cliente da API.
  client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=groq_api_key
  )
  return client_ai

def markdown_prepare(papel, categoria, contexto, dificuldade, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Nível de Dificuldade:
  Com valores de 1 a 4, onde 1 indica o mais simples e 4 o mais complexo.
  {dificuldade}

  ### Instrução:
  {instrucao}
  """

  return prompt_usuario

def questions_submit(client_ai, model_id, df_questions):
    # Criar lista vazia para guardar as respostas.
    responses = []

    # Repetição para percorrer todo Dataframe.
    for index, row in df_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      dificuldade = row['level']
      instrucao = row['turns']

      # Preencher prompt do usuário.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

      # Realizar consulta a IA.
      completion = client_ai.chat.completions.create(
          model= model_id,
          messages=[
            # Por questões de sintaxes informo a role, pois é um campo obrigatório,
            # porém o conteúdo é somente o do Markdown.
            {"role": "user", "content": prompt_usuario}
          ],
          max_tokens=1024,
          temperature=0.1 # Conservador.
      )
      response = completion.choices[0].message.content

      # Armazenar o resultado em uma lista.
      responses.append({"question": questao, "response": response})
      print(f"Questão {questao} processada com sucesso.")

    # Fechar conexão com a IA (somente se você a usou ativamente)
    client_ai.close()

    # por motivo de performance as repostas foram adicionadas a uma lista.
    # No retorno a lista é convertida para um dataframe.
    return pd.DataFrame(responses)

# Função para juntar dois Dataframes por um campo chave e opcionalmente selecionar colunas.
def merge_dataframes(df1, df2, key, columns=None):
  merged_df = pd.merge(df1, df2, on=key, how='inner')
  if columns is not None:
    return merged_df[columns]
  return merged_df

Realizar consulta usando o modelo llama 3 de 70 bilhões de parâmetros.

In [143]:
# Modelo llama 3 de 70 bilhões de parâmetros.
model_id = 'llama-3.3-70b-versatile'

# Dataframe com as respostas do primeiro modelo
df_llama_response = questions_submit(client_ai_instance('groq_api_key'), model_id, df_my_questions)

,question,llama3
0,31,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...
1,32,O prazo para oferta dos embargos à execução fi...
2,33,O Imposto Sobre Serviços (ISS) é um tributo mu...
3,34,A exigência de depósito do montante integral c...
4,35,O argumento apresentado pelo departamento jurí...
5,36,### RECURSO DE AGRAVO DE INSTRUMENTO\n\n**EXCE...
6,37,A responsabilização administrativa de uma soci...
7,38,A cobrança efetuada pelo órgão responsável par...
8,39,A possibilidade de utilização do sistema de re...
9,40,A aprovação de Iná no concurso público para a ...


Modelo llama 4 scout de 17 bilhões de parâmetros.

In [144]:
# Modelo llama 4 scout, mais leve, de 17 bilhões de parâmetros.
model_id = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Dataframe com as respostas do primeiro modelo
df_llama4_response = questions_submit(client_ai_instance('groq_api_key'), model_id, df_my_questions)

,question,llama4
0,31,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...
1,32,O prazo para oferta dos embargos à execução fi...
2,33,## Resposta\n\nO Imposto Sobre Serviços (ISS) ...
3,34,A exigência de depósito do montante integral c...
4,35,## Resposta\n\nO argumento da necessidade de n...
5,36,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...
6,37,A responsabilização administrativa de pessoas ...
7,38,A questão apresentada envolve a interpretação ...
8,39,"Sim, é possível a utilização do sistema de reg..."
9,40,A aprovação de Iná no concurso público para a ...


Específico para o gemini

In [145]:
# Função para usar modelos de IA específica ao Gemini.
def questions_gemini_submit(client_ai, model_id, df_questions):
  # Criar uma lista vazia, para guardar as respostas, por questão de performance.
  gemini_responses = []

  # Repetição que percorre todo Dataframe.
  for index, row in df_my_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      dificuldade = row['level']
      instrucao = row['turns']

      # Preparação do prompt em Markdown.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

      # Chamada simples para a API da Google em nuvem.
      response = client_ai.models.generate_content(
          model=model_id,
          contents=prompt_usuario,
          config={
              "temperature": 0.1,  # Conservador.
              "max_output_tokens": 1024
          }
      )

      # Armazenar o resultado em uma lista.
      gemini_responses.append({"question": questao, "response": response.candidates[0].content.parts[0].text})
      print(f"Questão {questao} processada com sucesso.")

  # Fechar conexão com a IA
  client_ai.close()

  # Para melhor visualização, converter as respostas em um Dataframe.
  return pd.DataFrame(gemini_responses)

Modelo gemini flash lançado em 2026 para ser mai leve.

In [146]:
# O modelo escolhi do para rodar é o Gemini da Google em nuvém preview de 2026.
model_id = 'gemini-3.1-flash-lite-preview'

# Dataframe com as respostas do primeiro modelo
df_gemini_response = questions_gemini_submit(client_ai_instance('google_api_key'), model_id, df_my_questions)

,question,gemini
0,31,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...
1,32,O prazo para a oferta dos Embargos à Execução ...
2,33,O Imposto Sobre Serviços de Qualquer Natureza ...
3,34,"Não, a exigência de depósito do montante integ..."
4,35,"Não, o argumento da sociedade empresária está ..."
5,36,EXCELENTÍSSIMO SENHOR DOUTOR DESEMBARGADOR PRE...
6,37,Para a caracterização da responsabilização adm...
7,38,"Não, a cobrança efetuada pelo órgão responsáve..."
8,39,"Sim, é possível a utilização do Sistema de Reg..."
9,40,Não. A aprovação de Iná no concurso público pa...


Arrumar as respostas das 3 IAs.

In [147]:
# Renomear colunas dos dataframes para melhor legibilidade.
df_llama_response.rename(columns={'response': 'llama3'}, inplace=True)
df_llama4_response.rename(columns={'response': 'llama4'}, inplace=True)
df_gemini_response.rename(columns={'response': 'gemini'}, inplace=True)
key = 'question'

# Realizar o primeiro inner join entre llama_responses e llama4_responses
df_llamas = merge_dataframes(df_llama_response, df_llama4_response, key)

# Realizar o segundo inner join com df_gemini_response
df_llamas_gemini = merge_dataframes(df_llamas, df_gemini_response, key)

# Junção com o Dataframe de respostas das IAs com o das perguntas, selecionando a coluna choices da linha de guia de respostas.
columns = ['question', 'statement', 'turns', 'llama3', 'llama4', 'gemini', 'choices']
df_ias = merge_dataframes(df_llamas_gemini, df_my_questions, key, columns)

Métrica usada para avaliar as respostas, BERTScore.

In [ ]:
# Função para calular a métrica, um valor entre 0 e 100%.
def calculate_bert_f1(candidates, references):
  if not isinstance(candidates, list) or not isinstance(references, list):
    print("Candidates e References precisam ser lista.")
    return []
  if len(candidates) != len(references):
    print("Candidates e References precisam ter a mesma quantidade de elementos.")
    return []

  # Conferir se os valores são strings.
  processed_candidates = []
  for item in candidates:
    if isinstance(item, (str, int, float)):
      processed_candidates.append(str(item))
    else:
      processed_candidates.append("")

  processed_references = []
  for item in references:
    if isinstance(item, (str, int, float)):
      processed_references.append(str(item))
    else:
      processed_references.append("")

  # Depois das validações, realizao o calculo propriamente dito.
  try:
    P, R, F1 = score(processed_candidates, processed_references, lang="pt", verbose=False)
    return [f * 100 for f in F1.tolist()]
  except Exception as e:
    print(f"Erro ao tentar calcular BERTScore: {e}")
    return []

Calcular a métrica BERTStore para o Dataframe das respostas das IAs.

In [149]:
# The dataframe df_ias is already available from previous steps.
df_ias['llama3_bert'] = calculate_bert_f1(df_ias['llama3'].tolist(), df_ias['choices'].tolist())
df_ias['llama4_bert'] = calculate_bert_f1(df_ias['llama4'].tolist(), df_ias['choices'].tolist())
df_ias['gemini_bert'] = calculate_bert_f1(df_ias['gemini'].tolist(), df_ias['choices'].tolist())

,question,statement,turns,llama3,llama4,gemini,choices,llama3_bert,llama4_bert,gemini_bert
0,31,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,"A medida judicial cabível é a ação anulatória,...",70.734304,71.521699,68.402725
1,32,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"A partir de quando se deve contar, no caso con...",O prazo para oferta dos embargos à execução fi...,O prazo para oferta dos embargos à execução fi...,O prazo para a oferta dos Embargos à Execução ...,O prazo para oferta dos embargos à execução fi...,61.487019,58.929962,64.990950
2,33,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,A qual dos municípios o ISS de jardinagem é de...,O Imposto Sobre Serviços (ISS) é um tributo mu...,## Resposta\n\nO Imposto Sobre Serviços (ISS) ...,O Imposto Sobre Serviços de Qualquer Natureza ...,O ISS de jardinagem é devido ao Município Beta...,55.021125,54.669273,53.596252
3,34,QUESTÃO\n\nA Administração Tributária Federal ...,É válida a exigência de depósito do montante c...,A exigência de depósito do montante integral c...,A exigência de depósito do montante integral c...,"Não, a exigência de depósito do montante integ...","Não, pois é inconstitucional a exigência de de...",56.042856,57.600617,60.703576
4,35,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,Está correto o argumento da necessidade de not...,O argumento apresentado pelo departamento jurí...,## Resposta\n\nO argumento da necessidade de n...,"Não, o argumento da sociedade empresária está ...","Não está correto, porque a entrega de declaraç...",54.082698,52.481550,54.668510
5,36,PEÇA PRÁTICO-PROFISSIONAL\n\nO Ministério Públ...,,### RECURSO DE AGRAVO DE INSTRUMENTO\n\n**EXCE...,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,EXCELENTÍSSIMO SENHOR DOUTOR DESEMBARGADOR PRE...,O(a) examinando(a) deve apresentar recurso de ...,70.651370,70.825714,69.228590
6,37,QUESTÃO\n\nA sociedade empresária Sagaz S.A. e...,Há necessidade de demonstração do elemento sub...,A responsabilização administrativa de uma soci...,A responsabilização administrativa de pessoas ...,Para a caracterização da responsabilização adm...,Não. A responsabilização administrativa por at...,55.595571,54.627019,56.706053
7,38,QUESTÃO\n\nDeterminada informação de interesse...,É lícita a cobrança efetuada pelo órgão respon...,A cobrança efetuada pelo órgão responsável par...,A questão apresentada envolve a interpretação ...,"Não, a cobrança efetuada pelo órgão responsáve...",Não. A submissão e o processamento de pedido d...,53.318810,54.296291,53.434801
8,39,QUESTÃO\n\nCerta Secretaria do Estado Alfa fez...,É possível a utilização do sistema de registro...,A possibilidade de utilização do sistema de re...,"Sim, é possível a utilização do sistema de reg...","Sim, é possível a utilização do Sistema de Reg...",Sim. A Administração poderá contratar a execuç...,52.865046,52.571273,58.502907
9,40,"QUESTÃO\n\nRecentemente, Iná foi aprovada em c...",A aprovação de Iná no mencionado concurso impo...,A aprovação de Iná no concurso público para a ...,A aprovação de Iná no concurso público para a ...,Não. A aprovação de Iná no concurso público pa...,"Não. Iná foi aprovada para emprego público, ao...",52.193022,56.177604,57.044733


Avaliar as respostas por um juiz on-line (IA).

In [157]:
def llm_judge(client_ai, model_id, name_ia):
  # Criar lista vazia para guardar as respostas.
  responses = []

  # Repetição para percorrer todo Dataframe.
  for index, row in df_ias.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    pergunta = row['statement'] + ' ' + row['turns']
    baseline = row['choices']
    ia = row[name_ia]

    prompt_usuario = f"""
    Você é um jurista sênior especializado em Direito Administrativo e Tributário brasileiro.
    Sua função é atuar como juiz em um benchmark de IAs.
    Analise as respostas comparando-as rigorosamente com a Resposta Base (Gabarito).
    Atribua um percentual de acerto (0-100%). Traga apensa um número com duas casas decimais.
    PERGUNTA: {pergunta}

    RESPOSTA BASE (GABARITO): {baseline}
    ---
    RESPOSTA IA: {ia}
      ---
    [FORMATO DE RESPOSTA]
    Apenas um número com duas casas decimais.
    """
    # Realizar consulta a IA.
    completion = client_ai.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.1 # Conservador
    )
    response = completion.choices[0].message.content

    # Armazenar o resultado em uma lista.
    responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

  # Fechar conexão com a IA (somente se você a usou ativamente)
  client_ai.close()

  # por motivo de performance as repostas foram adicionadas a uma lista.
  # No retorno a lista é convertida para um dataframe.
  return pd.DataFrame(responses)

In [160]:
model_id = "nvidia/nemotron-3-nano-30b-a3b:free"
df_score_lama3 = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'llama3')
df_score_lama4 = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'llama4')
df_score_gemini = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'gemini')

,question,response
0,31,None
1,32,100.00
2,33,0.00
3,34,84.62
4,35,100.00
5,36,38.00
6,37,100.00
7,38,100.00
8,39,84.62
9,40,0.00
